# Step 03.7 — Reduction, axis=-1, keepdims

这一节把 3 个紧密相关的概念一起学完：

1. reduction：把一组数压缩成更少的数；
2. `axis=-1`：最后一个轴；
3. `keepdims=True`：压缩后保留这个轴，方便后续 broadcasting。

In [ ]:
import mlx.core as mx

x = mx.array([
    [1.0, 2.0, 3.0],
    [4.0, 5.0, 6.0],
])

print(x)
print("shape:", x.shape)


## 1. Reduction 是什么？

`sum`、`mean` 这类操作会把某个 axis 上的一组数字压缩成一个数字。

例如 `x.shape = (2, 3)`：

- `axis=0`：沿着两行压缩，最后留下 3 个数；
- `axis=1`：沿着每一行的 3 个元素压缩，最后留下 2 个数。

In [ ]:
mean_axis0 = mx.mean(x, axis=0)
mean_axis1 = mx.mean(x, axis=1)

print("mean axis=0:", mean_axis0, "shape:", mean_axis0.shape)
print("mean axis=1:", mean_axis1, "shape:", mean_axis1.shape)


手算：

`axis=0`：

- `(1 + 4) / 2 = 2.5`
- `(2 + 5) / 2 = 3.5`
- `(3 + 6) / 2 = 4.5`

所以得到 `[2.5, 3.5, 4.5]`，shape 从 `(2, 3)` 变成 `(3,)`。

`axis=1`：每一行各算一个平均数，所以得到 `[2.0, 5.0]`，shape 变成 `(2,)`。

## 2. `axis=-1` 是什么？

Python/MLX 允许从右往左编号 axis：

```text
shape = (B, T, C)
         0  1  2
        -3 -2 -1
```

所以 `axis=-1` 永远代表**最后一个轴**。

这在神经网络里很方便，因为最后一维通常就是 feature/channel 维。

In [ ]:
a = mx.mean(x, axis=-1)
b = mx.mean(x, axis=1)

print("axis=-1:", a)
print("axis=1 :", b)
print("same result:", bool(mx.array_equal(a, b)))


## 3. 为什么需要 `keepdims=True`？

默认 reduction 会把被压缩的 axis 删除。

例如：

```text
(2, 3) --mean(axis=-1)--> (2,)
```

但如果加：

```python
keepdims=True
```

则会变成：

```text
(2, 3) --mean(axis=-1, keepdims=True)--> (2, 1)
```

也就是最后这个 axis 没消失，只是长度从 3 变成 1。

In [ ]:
m1 = mx.mean(x, axis=-1)
m2 = mx.mean(x, axis=-1, keepdims=True)

print("without keepdims:", m1, "shape:", m1.shape)
print("with keepdims   :", m2, "shape:", m2.shape)


## 4. keepdims 为什么对 RMSNorm 很重要？

假设每一行代表一个 token 的 3 个特征：

```text
[1, 2, 3]
[4, 5, 6]
```

如果我们想让每一行都除以自己的平均值，就希望得到：

```text
mean.shape = (2, 1)
```

这样 broadcasting 会自动把每一行对应的那个平均值扩展到 3 个特征。

In [ ]:
row_mean = mx.mean(x, axis=-1, keepdims=True)
normalized_demo = x / row_mean

print("row_mean:")
print(row_mean)
print("row_mean shape:", row_mean.shape)

print("\nx / row_mean:")
print(normalized_demo)


这里发生的是：

```text
x        : (2, 3)
row_mean : (2, 1)
```

broadcasting 后，每一行用自己的那个标量去除。

这就是以后 RMSNorm 中：

```python
mx.mean(..., axis=-1, keepdims=True)
```

背后的 shape 逻辑。

## Step 3 总结

到这里你已经具备后面读 Transformer shape 所需的最小张量基础：

- tensor / shape
- axis
- indexing
- reshape
- transpose
- broadcasting
- reduction
- `axis=-1`
- `keepdims=True`

下一步开始进入真正的神经网络计算。